# Text Simplification Pipeline for ClearSim using local GPU

In [1]:
import pandas as pd
import torch
import numpy as np

In [2]:
# change to each experiment ID
exp_id = "{Exp_ID}"

In [ ]:
#Loading in ClearSim dataset

#150 words roughly equate to 200 tokens from https://platform.openai.com/tokenizer

# sampling paragraphs that have a max on 300 words
def load_clears_paired(csv_path, max_orig_words=300):
    df = pd.read_csv(csv_path)
    txt = df[df['type'] == 'TXT'][['row_id', 'text']].rename(columns={'text': 'original'})
    fac = df[df['type'] == 'FAC'][['row_id', 'text']].rename(columns={'text': 'simplified'})
    paired = txt.merge(fac, on='row_id')

    paired['orig_words'] = paired['original'].str.split().str.len()
    paired = paired[paired['orig_words'] <= max_orig_words].drop(columns='orig_words')
    return paired.reset_index(drop=True)

# Use the provided train/test splits
data_train = load_clears_paired('/home/c23068554/final_project/datasets/cleartext-discriminativo-es/train.csv')
data_test = load_clears_paired('/home/c23068554/final_project/datasets/cleartext-discriminativo-es/test.csv')

print(f"Train pairs (filtered): {len(data_train)}")
print(f"Test pairs (filtered): {len(data_test)}")

In [ ]:
from sklearn.model_selection import train_test_split

data_test, _ = train_test_split(data_test, train_size=134, random_state=59)
print(f"Subsampled to {len(data_test)} entries")

In [5]:
# selection for Few-shot count
random_state = 59
examples = data_train.sample(n=3, random_state = random_state)

## Prompt and Few-shot construction



### Prompt (Spanish)

In [ ]:
BLESS_2 = "Por favor, reformula el siguiente texto complejo para que sea más comprensible para hablantes no nativos de español. Puedes hacerlo reemplazando palabras complejas por sinónimos más sencillos (parafraseando), eliminando información irrelevante (condensando) o dividiendo la oración en varias más simples. La oración simplificada final debe ser gramaticalmente correcta, fluida y conservar las ideas principales del original sin alterar su significado.\n\n"

### Prompt (English)

In [ ]:
BLESS_2_Eng = "Please rewrite the following complex text in order to make it easier to understand by non-native speakers of Spanish. You can do so by replacing complex words with simpler synonyms (i.e. paraphrasing), deleting unimportant information (i.e. compression), and/or splitting a long complex sentence into several simpler ones. The final simplified sentence needs to be grammatical, fluent, and retain the main ideas of its original counterpart without altering its meaning.\n\n"

In [ ]:
instruction = "{SELECT PROMPT}"
def makePrompt(instruction, examples):
  #formatting text for fewshot examples
  fewshot = ""
  for index, row in examples.iterrows():
    fewshot += (f"Compleja: {row.loc['original']}\nSimplificada: {row.loc['simplified']}\n\n")
  return(instruction + fewshot)

fewshot_example = makePrompt(instruction, examples)

## Load in model and inference - GPU

In [ ]:
# Test for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"compute type: {device}")

### Decoder models

In [8]:
# Decoder models
minstral_7b = "mistralai/Mistral-7B-Instruct-v0.3"
llama31_8b = "meta-llama/Meta-Llama-3.1-8B-Instruct"
phi3_128k = "microsoft/Phi-3-medium-128k-instruct"
gemma2_9b = "google/gemma-2-9b-it"
qwen2_7b = "Qwen/Qwen2.5-7B-Instruct"
llama33_70b = "meta-llama/Meta-Llama-3.1-70B-Instruct"

In [9]:
from transformers import AutoModelForCausalLM, AutoTokenizer

#tokenizer = AutoTokenizer.from_pretrained("{MODEL_ID}")
#model = AutoModelForCausalLM.from_pretrained("{MODEL_ID}").to(device)

In [10]:
def generateDec(prompt):
    messages = [{"role": "user", "content": prompt}]
    input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to(device)

    # Store input length so we can slice it off later
    input_len = input_ids.shape[1]

    output = model.generate(
        input_ids,
        #example on HF sets this as 1000
        max_new_tokens=1000
    )

    # Decoder output includes the input — slice it off
    return tokenizer.decode(
        output[0][input_len:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    ).strip()

### Encoder-Decoder models

In [11]:
# Encoder-Decoder Models
flanT5small = "google/flan-t5-small"
flanT5large = "google/flan-t5-large"
flanT5xl = "google/flan-t5-xl" #3B parameters
mT5 = "google/mt5-large" #1.2B parameters 

In [ ]:
from transformers import AutoTokenizer, T5ForConditionalGeneration, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("{MODEL_ID}")
model = T5ForConditionalGeneration.from_pretrained("{MODEL_ID}").to(device)

In [13]:
def generateEncDec(prompt):
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    output = model.generate(
        input_ids,
        max_length = 512,
        #do_sample=False,
        #num_beams=1,
        #encoder_no_repeat_ngram_size=5
    )
    return tokenizer.decode(
        output[0],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    ).strip()

In [ ]:
print(model.generation_config)

## Looping through dataset with prompt

In [14]:
#looping through all n-shot prompt
def promptLoop(fewshot_example, data_test):
  LMsimplified = []
  for i, row in enumerate(data_test['original']):
    full_prompt = fewshot_example + f"Compleja: {row}\nSimplificada:"
    LMsimplified.append(generateEncDec(full_prompt))
    #progress check on infference
    if (i + 1) % 10 == 0:
            print(f"  {i+1}/{len(data_test)} done")
  return LMsimplified

### Inference

In [ ]:
LMoutput = (promptLoop(fewshot_example, data_test))

### Organising Outputs

In [17]:
reference = data_test['simplified'].tolist()
source = data_test['original'].tolist()

In [ ]:
import csv

out_path = "lm_output.csv"

with open(out_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(["Exp ID", "Index", "Original", "Reference", "LM Output"])
    for i, (orig, ref, lm) in enumerate(zip(source, reference, LMoutput)):
        writer.writerow([exp_id, i, orig, ref, lm])

print(f"Wrote {len(source)} rows to {out_path}")

# Evaluation

### Copy sentences


In [ ]:
count = 0
for original, simple, lmSimple in zip(data_test['original'], data_test['simplified'], LMoutput):
  print(original)
  print(simple)
  print(lmSimple + "\n")
  if original == lmSimple:
    #print(original)
    #print(simple)
    #print(lmSimple)
    count += 1
print(f"Number of sentences not simplified or altered by the model: {count}")

In [ ]:
import evaluate

#load metrics
rouge = evaluate.load('rouge')
bleu = evaluate.load('bleu')
bertscore = evaluate.load('bertscore')
sari = evaluate.load('sari')

#Converting simplified reference sentences into a list inside a list
sari_references = [[s] for s in data_test["simplified"].astype(str).tolist()]
sari_score = sari.compute(sources= source, predictions= LMoutput, references= sari_references)

# Compute scores
rouge_results = rouge.compute(predictions= LMoutput, references = reference)
bleu_results = bleu.compute(predictions= LMoutput, references = reference)

bertscore_compute = bertscore.compute(predictions=LMoutput, references= reference, lang='es', model_type = 'xlm-roberta-large')
berstcoreAvg = np.mean(bertscore_compute['f1'])

### Fernandes Huerta Score

In [23]:

#fernandex_huerta score
import textstat
textstat.set_lang("es")


fhOrig = np.mean([textstat.fernandez_huerta(s) for s in data_test["original"].tolist()])
fhSimp = np.mean([textstat.fernandez_huerta(s) for s in LMoutput])

In [ ]:
#output
print(f"Dataset: CLEARS (FAC), size: {len(data_test)}, random state: {random_state}")
print()
print(f"ROUGE Score: {rouge_results['rouge1']}")
print(f"BLEU Score: {bleu_results['bleu']}")
print(f"BERTScore Score (xlm-roberta-large): {berstcoreAvg}")
print(f"Sari Score: {sari_score['sari']}")
print()
print(f"Original Fernández-Huerta score: {fhOrig}")
print(f"Simplified Fernández-Huerta score: {fhSimp}")
print()
print(f"unsimplified sentences: {count}/{len(data_test)}")

In [ ]:
print(f"\nTAB-SEPARATED (paste into experiment sheet metrics columns):")
print(f"{sari_score['sari']:.4f}\t{rouge_results['rouge1']:.4f}\t{bleu_results['bleu']:.4f}\t{berstcoreAvg:.4f}\t{fhSimp:.4f}\t{count}/{len(data_test)}")

In [ ]:
# command to clear cache often, to reduce disk space used:
# rm -rf ~/.cache/*

#check disk space used:
# du -h --max-depth=1 ~ | sort -h

'''
Exporting CSV:
On local:
scp -r c23068554@10.98.84.2:/home/c23068554/final_project/lm_output.csv '/Users/justinwoodham/Desktop/CS/Y3/Final Year Project/raw outputs'
On Remote: 
rm lm_output.csv

'''